In [ ]:
import tensorflow as tf

def build_keras_pipeline(file_paths, labels, batch_size=32):
    # Create dataset from tensor slices
    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    
    def process_path(file_path, label):
        img = tf.io.read_file(file_path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32) # Normalization [0,1]
        img = tf.image.resize(img, [224, 224])
        return img, label

    # Optimization Pipeline
    ds = ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    
    # Augmentation: Essential for satellite invariance
    ds = ds.map(lambda x, y: (tf.image.random_flip_left_right(x), y))
    ds = ds.map(lambda x, y: (tf.image.random_brightness(x, 0.2), y))
    
    ds = ds.cache().shuffle(1000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# Critical Insight: .prefetch(tf.data.AUTOTUNE) overlaps the 
# preprocessing and model execution of a training step.